# Relative permeability hysteresis: the PUNQ-S3 case

This notebook reproduces Section 5 of `report/report.tex` (the paper's
Figs. 3 and 4): the Killough/Land hysteresis model for gas relative
permeability, and its validation on the PUNQ-S3 geologic model.

## Killough's model with Land trapping

Relative permeability depends on the saturation path, not only on the
current saturation: pore-scale contact-angle hysteresis and nonwetting
phase disconnection make the imbibition curve differ from the drainage
curve. In geologic CO2 storage, CO2 is the nonwetting phase in most
aquifers; during post-injection migration, brine re-imbibes at the
trailing edge of the plume, disconnects CO2 ganglia, and immobilizes them
(residual trapping). A drainage-only model misses this and overpredicts
the mobile gas cap.

Killough's model computes the scanning curve between the primary drainage
curve $k^d_{rg}$ and the bounding imbibition curve $k^{ib}_{rg}$. While the
current saturation $S_g$ is below the historical maximum $S_{gi}$,
$$
k^{i}_{rg}(S_g) = k^{ib}_{rg}(S_g^*)\,
\frac{k^{d}_{rg}(S_{gi})}{k^{d}_{rg}(S_{g,\max})},
\qquad
S_g^* = S_{gt,\max} + \frac{(S_g - S_{gt})(S_{g,\max} -
S_{gt,\max})}{S_{gi} - S_{gt}},
$$
where $S_{g,\max}$ is the maximum attainable gas saturation and
$S_{gt,\max}$ the corresponding maximum trapped saturation. The trapped
gas saturation for the current cycle follows Land's model,
$$
S_{gt} = S_{g,\min} + \frac{S_{gi} - S_{g,\min}}
{1 + C\,(S_{gi} - S_{g,\min})},
\qquad
C = \frac{1}{S_{gt,\max} - S_{g,\min}} - \frac{1}{S_{g,\max} -
S_{g,\min}},
$$
with $S_{g,\min}=0$ on primary drainage. MRST reads the hysteresis options
from the deck's `EHYSTR` keyword and evaluates this inside a dedicated
state function, `HystereticRelativePermeability`, that replaces the
default relative permeability of the validated model.

In [ ]:
run('/home/adriano/codes/MRST/startup.m');
mrstModule add ad-props deckformat ad-core ad-blackoil
addpath('/home/adriano/codes/MRST/core/utils/octave_only/mrst');
gravity reset on
mrstVerbose off

RESDIR = '/home/adriano/codes/MRST/reproductions/salo2024-gcs/results';
FIGDIR = '/home/adriano/codes/MRST/reproductions/salo2024-gcs/figures';
DECKDIR = '/home/adriano/codes/MRST/mrst-adblackoil-gcs/kr-hysteresis';

## Hysteresis curves for the PUNQ-S3 rock

`addScanKr` (from the paper's companion repository) builds the scanning
curve for a given flow reversal. Its directory is added at the **end** of
the Octave path (`'-end'`): it also contains a *different*,
paper-specific `HystereticRelativePermeability.m` that must not shadow
MRST's own state function used by the simulation cells below.

In [ ]:
fn = fullfile(DECKDIR, 'input_files_kr/punq-s3/CASE2.DATA');
deck  = convertDeckUnits(readEclipseDeck(fn));
fluid = initDeckADIFluid(deck);
fluid.krHyst = 2;

addpath(fullfile(DECKDIR, 'code'), '-end');   % addScanKr only; see note above
G    = computeGeometry(initEclipseGrid(deck));
rock = compressRock(initEclipseRock(deck), G.cells.indexMap);
fluid = addScanKr(fluid, rock.regions.imbibition, 0.02);

sgd = linspace(fluid.krPts.g(1,2), fluid.krPts.g(1,3), 50)';
krd = fluid.krG{1}(sgd);                       % primary drainage
sgi = linspace(fluid.krPts.g(2,2), fluid.krPts.g(2,3), 50)';
kri = fluid.krG{2}(sgi);                       % bounding imbibition
sgs = linspace(0.28, 0.4, 50)';
krs = fluid.krGi{1}(sgs, repelem(0.4, 50, 1)); % scanning curve, sg_max=0.4

figure('Position', [100 100 480 400]);
plot(sgd, krd, '-r', 'linewidth', 2); hold on
plot(sgi, kri, '--b', 'linewidth', 2);
plot(sgs, krs, '-.k', 'linewidth', 1.5); hold off
grid on
xlabel('S_g [-]'), ylabel('k_{rg} [-]')
legend('drainage k_{rg}^d', 'bounding imbibition k_{rg}^{ib}', ...
       'scanning curve (S_{gi}=0.4)', 'location', 'northwest')
xlim([0 1]), ylim([0 1])
fprintf('Land trapped saturation at reversal Sgi=0.4: Sgt = %.3f\n', min(sgs(krs<=1e-6)));
print(gcf, fullfile(FIGDIR, 'fig_kr_hysteresis.png'), '-dpng', '-r150');

## Validation on the PUNQ-S3 model

Juanes et al. (2006) adapted the PUNQ-S3 geologic model into a GCS
benchmark: a $19\times28\times5$ corner-point grid (1,761 active cells)
of an anticline aquifer with heterogeneous permeability (0.5-1000 mD),
eight injectors at 18 rm3/day each for 10 years, 500 years of total
simulation, immiscible water-gas physics, and pore volumes multiplied by
$10^3$ on the boundary ring to mimic an open aquifer. Both published cases
come from the same deck (`CASE2.DATA`): Case 1 deactivates hysteresis;
Case 2 activates Killough hysteresis in the gas phase.

Set `RUN_LIVE = true` below to re-run both 500-year cases from scratch
(about 34 s and 51 s). By default this notebook loads the cached results
in `results/punq_case1_nohyst.mat` and `results/punq_case2_hyst.mat`.

In [ ]:
RUN_LIVE = false;

In [ ]:
function state0 = punq_state0(G, fluid)
    g = norm(gravity);
    z_0 = min(G.cells.centroids(:,3));
    z_max = max(G.cells.centroids(:,3));
    p_r = g*fluid.rhoWS*z_0;
    nz = 2000;
    zv = linspace(z_0, z_max, nz)';
    dz = zv(2) - zv(1);
    pv_hydro = zeros(nz,1);  pv_hydro(1) = p_r;
    for k = 2:nz
        dpdz = g * fluid.bW(pv_hydro(k-1)) * fluid.rhoWS;
        pmid = pv_hydro(k-1) + 0.5*dz*dpdz;      % midpoint rule
        pv_hydro(k) = pv_hydro(k-1) + dz * g * fluid.bW(pmid) * fluid.rhoWS;
    end
    p0 = interp1(zv, pv_hydro, G.cells.centroids(:,3));
    s0 = repmat([1, 0], [G.cells.num, 1]);       % fully saturated in water
    state0 = struct('s', s0, 'rs', 0, 'rv', 0, 'pressure', p0);
end

In [ ]:
if RUN_LIVE
  cases = struct('krhyst', {0, 1}, 'outtag', {'case1_nohyst', 'case2_hyst'});
  fnpunq = fullfile(DECKDIR, 'input_files_kr/punq-s3/CASE2.DATA');
  for ci = 1:numel(cases)
    krhyst = cases(ci).krhyst;
    outtag = cases(ci).outtag;

    deck = convertDeckUnits(readEclipseDeck(fnpunq));
    G = initEclipseGrid(deck);
    shiftZup = 840 - min(G.nodes.coords(:, 3));
    G.nodes.coords(:, 3) = G.nodes.coords(:, 3) + shiftZup;
    G = computeGeometry(G);

    rock  = initEclipseRock(deck);
    rock  = compressRock(rock, G.cells.indexMap);

    fluidC = initDeckADIFluid(deck);
    rock.regions.saturation = ones(G.cells.num, 1);
    if krhyst == 1
        fluidC.krHyst = 2;
    else
        fluidC = rmfield(fluidC, 'krHyst');
    end

    model = selectModelFromDeck(G, rock, fluidC, deck);
    model = model.validateModel();
    nls = getNonLinearSolver(model, 'TimestepStrategy', 'iteration');
    nls.LinearSolver = BackslashSolverAD();
    nls.useLinesearch = true;
    nls.maxIterations = 15;
    nls.maxTimestepCuts = 12;
    nls.acceptanceFactor = 2;

    idAct = find(deck.GRID.ACTNUM);
    idG = (1:prod(deck.GRID.cartDims))';
    idG_mult = idG(deck.GRID.PORV==1000);
    cellsb = ismember(idAct, idG_mult);
    model.operators.pv(cellsb) = model.operators.pv(cellsb)*1e3;
    model.FlowPropertyFunctions = ...
    model.FlowPropertyFunctions.setStateFunction('RelativePermeability', ...
                                                 HystereticRelativePermeability(model));

    state0 = punq_state0(G, fluidC);
    schedule = convertDeckScheduleToMRST(model, deck);
    resv = 18/day;
    pavg = mean(state0.pressure([schedule.control(1).W.cells]));
    rate = resv*fluidC.bG(pavg);
    [schedule.control(1).W.val] = deal(rate);
    [schedule.control(2).W.val] = deal(0);
    [schedule.control(1).W.type] = deal('rate');
    [schedule.control(2).W.type] = deal('rate');
    nwell = numel(schedule.control(1).W);
    for n=1:nwell
        schedule.control(1).W(n).lims.bhp = 160*barsa;
    end

    t_start = tic;
    [wellSols, states, report] = simulateScheduleAD(state0, model, schedule, ...
                                                    'NonLinearSolver', nls);
    t_elapsed = toc(t_start);
    fprintf('%s: done in %.1f s (%d steps)\n', outtag, t_elapsed, numel(states));

    ns = numel(states);
    sg_all = zeros(G.cells.num, ns);
    p_all  = zeros(G.cells.num, ns);
    for n = 1:ns
        sg_all(:,n) = states{n}.s(:,2);
        p_all(:,n)  = states{n}.pressure;
    end
    tvec = cumsum(schedule.step.val);
    indexMap = G.cells.indexMap;
    cartDims = deck.GRID.cartDims;
    p0 = state0.pressure;
    save('-v7', fullfile(RESDIR, ['punq_' outtag '.mat']), ...
         'sg_all', 'p_all', 'tvec', 'indexMap', 'cartDims', 'p0', 't_elapsed');
  end
end

c1 = load(fullfile(RESDIR, 'punq_case1_nohyst.mat'));
c2 = load(fullfile(RESDIR, 'punq_case2_hyst.mat'));
fprintf('Loaded. Runtimes: case1 %.1f s, case2 %.1f s\n', c1.t_elapsed, c2.t_elapsed);

## CO2 saturation maps at t = 500 y

Both cases coincide until injection stops at $t=10$ y, because hysteresis
only acts on saturation reversals. Compare with Fig. 4 of the report.

In [ ]:
nx = c1.cartDims(1); ny = c1.cartDims(2); nz = c1.cartDims(3);

% ECLIPSE-like colormap
colors = [0 0 1; 0 1 1; 0 1 0; 1 1 0; 1 0 0];
cmap = [interp1([0 1], colors(1:2,:), linspace(0,1,30));
        interp1([0 1], colors(2:3,:), linspace(0,1,25));
        interp1([0 1], colors(3:4,:), linspace(0,1,25));
        interp1([0 1], colors(4:5,:), linspace(0,1,20))];

cases2 = {c1, c2};
names = {'No hysteresis', 'Hysteresis'};
figure('Position', [50 50 1500 550]);
for c = 1:2
    d = cases2{c};
    sgc = -0.011*ones(nx*ny*nz, 1);        % inactive-cell sentinel, shown gray
    sgc(d.indexMap) = d.sg_all(:, end);    % final state, t = 500 y
    sg3 = reshape(sgc, nx, ny, nz);
    for k = 1:nz
        subplot(2, nz, (c-1)*nz + k);
        imagesc(squeeze(sg3(:,:,k))');
        axis equal tight; caxis([-0.011 1]); colormap([0.85 0.85 0.85; cmap]);
        set(gca, 'XTick', [], 'YTick', []);
        if c == 1, title(sprintf('Layer %d', k)); end
        if k == 1, ylabel(names{c}); end
    end
end
subplot(2, nz, 2*nz); colorbar;
print(gcf, fullfile(FIGDIR, 'fig_punq_sg500y.png'), '-dpng', '-r130');

## Plume metrics

Without hysteresis the plume migrates up-dip and collects into a compact,
high-saturation gas cap. With hysteresis the trailing edge traps CO2
residually along the migration path, and the mean plume saturation drops.

In [ ]:
fprintf('%-28s %-14s %-14s\n', 'Metric (t = 500 y)', 'No hysteresis', 'Hysteresis');
fprintf('%-28s %-14.3f %-14.3f\n', 'Max Sg', ...
        max(c1.sg_all(:,end)), max(c2.sg_all(:,end)));
fprintf('%-28s %-14d %-14d\n', 'Cells with Sg > 0.05', ...
        sum(c1.sg_all(:,end) > 0.05), sum(c2.sg_all(:,end) > 0.05));
fprintf('%-28s %-14d %-14d\n', 'Cells with Sg > 0.3', ...
        sum(c1.sg_all(:,end) > 0.3), sum(c2.sg_all(:,end) > 0.3));
fprintf('%-28s %-14.3f %-14.3f\n', 'Mean Sg where Sg > 0.05', ...
        mean(c1.sg_all(c1.sg_all(:,end)>0.05, end)), ...
        mean(c2.sg_all(c2.sg_all(:,end)>0.05, end)));

fprintf('\n%-28s %-14s %-14s\n', 'Metric (t = 10 y)', 'No hysteresis', 'Hysteresis');
fprintf('%-28s %-14.3f %-14.3f\n', 'Max Sg', ...
        max(c1.sg_all(:,10)), max(c2.sg_all(:,10)));
fprintf('%-28s %-14d %-14d\n', 'Cells with Sg > 0.05', ...
        sum(c1.sg_all(:,10) > 0.05), sum(c2.sg_all(:,10) > 0.05));

## Plume evolution

The curves coincide during injection ($t\le10$ y) and diverge after flow
reversal, when the hysteretic scanning curves activate.

In [ ]:
ty = c1.tvec/365.25/86400;                     % years
n1 = sum(c1.sg_all > 0.05, 1);                 % plume extent (cells)
n2 = sum(c2.sg_all > 0.05, 1);
m1 = max(c1.sg_all, [], 1);                    % max saturation
m2 = max(c2.sg_all, [], 1);

figure('Position', [100 100 900 360]);
subplot(1,2,1)
semilogx(ty, n1, '-r', 'linewidth', 2); hold on
semilogx(ty, n2, '-b', 'linewidth', 2); hold off
grid on
xlabel('t [years]'), ylabel('cells with S_g > 0.05')
legend('no hysteresis', 'hysteresis', 'location', 'northwest')
subplot(1,2,2)
semilogx(ty, m1, '-r', 'linewidth', 2); hold on
semilogx(ty, m2, '-b', 'linewidth', 2); hold off
grid on
xlabel('t [years]'), ylabel('max S_g [-]')
print(gcf, fullfile(FIGDIR, 'fig_punq_evolution.png'), '-dpng', '-r150');